In [82]:
import numpy as np
import pandas as pd
from scipy.sparse import dok_matrix, save_npz, diags
import os
import json

In [83]:
# Load customized disease_to_snomed_id
with open("../Data/disease_to_snomed_id.json",'r') as f:
    disease_to_snomed_id = json.load(f)

In [84]:
DISEASE = input("Disease: ")
OUTPUT_FOLDER = f"./output/DGIDB_{DISEASE}/"
# Leave blank for the all drugs
SNOMED_DISEASE_CODES = [disease_to_snomed_id[DISEASE]] #choose the corresponding SNOMED id for the disease

In [85]:
print(SNOMED_DISEASE_CODES)

[49049000]


In [86]:
def get_snomed_id(term: str) -> str:
    """
    Query SNOMED CT Snowstorm API and return the SNOMED concept ID 
    for a given disease/medical term.

    Raises KeyError if the term is not found.
    """
    url = (
        "https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/"
        "MAIN/concepts?limit=10&term={}"
    ).format(term)

    r = requests.get(url)
    if r.status_code != 200:
        raise RuntimeError(f"SNOMED API error: {r.status_code}")

    data = r.json()
    items = data.get("items", [])

    if not items:
        raise KeyError(f"SNOMED ID not found for term '{term}'")

    # Take the best match (first result)
    concept_id = items[0]["conceptId"]
    return concept_id

In [87]:
DGIDB = pd.read_csv("../Data/DGIDB/DrugToGene.tsv", sep="\t")
HUMANNET = pd.read_csv("../Data/HumanNet/HumanNet-GSP.tsv", sep="\t")
DDDB = pd.read_csv("../Data/DDDB/DrugToDisease_DGIDB_naming.tsv", sep="\t")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [88]:
DDDB

,NDF-RT,SNOMED,ndfrt_preferred_label,snomed_disease
0,N0000004713,26929004,DONEPEZIL,Alzheimer's disease
1,N0000004713,56267009,DONEPEZIL,Multi-infarct dementia
2,N0000004713,80098002,DONEPEZIL,Diffuse Lewy body disease
3,N0000004713,386806002,DONEPEZIL,Impaired cognition (finding)
4,N0000004713,425390006,DONEPEZIL,Dementia associated with Parkinson's Disease (...
...,...,...,...,...
3534,N0000146103,19905009,SULFAMETHOXAZOLE,Chronic prostatitis
3535,N0000146103,423561003,SULFAMETHOXAZOLE,Community-acquired methicillin resistant Staph...
3536,N0000146103,429271009,SULFAMETHOXAZOLE,Ventilator-acquired pneumonia (disorder)
3537,N0000146103,192701001,SULFAMETHOXAZOLE,Toxoplasma encephalitis


In [89]:
#extracting the drugs related to that specific disease
specific_disease_drugs = DDDB.loc[DDDB['SNOMED'].isin(SNOMED_DISEASE_CODES), 'ndfrt_preferred_label'].dropna().unique().tolist()
print(specific_disease_drugs)

['LEVODOPA', 'CLOZAPINE', 'CARBIDOPA ANHYDROUS']


In [90]:
# Filter only the relevant genes
if specific_disease_drugs:
    relevant_rows = DGIDB[DGIDB['drug_name'].isin(specific_disease_drugs)].copy()
else:
    relevant_rows = DGIDB.copy()

In [91]:
list(relevant_rows["gene_claim_name"])

['DRD2',
 'DRD2',
 'MAPK1',
 'SRC',
 'CYP3A43',
 'ABCG2',
 'CYP1A1',
 'NCBIGENE:12',
 'HLA-C',
 'MAOB',
 'ADA',
 'NCBIGENE:262',
 'GSTT1',
 'HTR2A',
 'MAOB',
 'NR4A1',
 'SH2B1',
 'C3',
 'UGT2B10',
 'ABCC1',
 'HLA-DRB5',
 'GRIN2B',
 'CYP2C18',
 'HTR2A',
 'UGT1A1',
 'UNIPROT:P20711',
 'NCBIGENE:1',
 'NCBIGENE:3',
 'NCBIGENE:11',
 'GLP1R',
 'AGTR1',
 'DLG4',
 'GFRA2',
 'GDNF',
 'MTHFR',
 'CYP2C19',
 'BDNF',
 'NT5E',
 'HTR1A',
 'NFIB',
 'HLA-DRB3',
 'NCBIGENE:24',
 'NCBIGENE:215',
 'NCBIGENE:217',
 'UNIPROT:P21728',
 'UNIPROT:P21918',
 'HTR3A',
 'GSTM1',
 'CYP2D6',
 'NCBIGENE:265',
 'MIR1912',
 'FAAH',
 'CYP3A4',
 'DRD1',
 'CYP3A4',
 'BAX',
 'HTR7',
 'MIR1264',
 'CYP3A4',
 'AOC1',
 'CHRM1',
 'TRAC',
 'NCBIGENE:22',
 'NCBIGENE:218',
 'PIK3CG',
 'NCBIGENE:8',
 'HTR7',
 'POLI',
 'FASN',
 'IL2',
 'ELAVL2',
 'CCKBR',
 'PRKAB2',
 'HLA-DPB1',
 'MC4R',
 'NCBIGENE:23',
 'SLC1A1',
 'LEP',
 'ITIH3',
 'TBC1D1',
 'UNIPROT:P21728',
 'IL15',
 'PRL',
 'SLC1A1',
 'GCG',
 'DTNBP1',
 'HLA-B',
 'TRAT1',
 'RAB

In [92]:
# Create mappings for vertices and hyperedges
relevant_rows['ncbi_gene_id'] = relevant_rows['ncbi_gene_id'].astype(str)
genes = relevant_rows['ncbi_gene_id'].unique()
drugs = relevant_rows['drug_name'].unique()
gene_to_index = {gene: i for i, gene in enumerate(genes)}
drug_to_index = {drug: i for i, drug in enumerate(drugs)}
# Define file paths
gene_to_index_path = OUTPUT_FOLDER + f"gene_to_index.json"
drug_to_index_path = OUTPUT_FOLDER + f"drug_to_index.json"

# Save gene_to_index mapping
with open(gene_to_index_path, 'w') as gene_file:
    json.dump(gene_to_index, gene_file, indent=4)

# Save drug_to_index mapping
with open(drug_to_index_path, 'w') as drug_file:
    json.dump(drug_to_index, drug_file, indent=4)

print(f"Mappings saved to {gene_to_index_path} and {drug_to_index_path}.")

Mappings saved to ./output/DGIDB_PARKINSON/gene_to_index.json and ./output/DGIDB_PARKINSON/drug_to_index.json.


In [93]:
relevant_rows

,gene_claim_name,gene_concept_id,gene_name,interaction_source_db_name,interaction_source_db_version,interaction_type,interaction_score,drug_claim_name,drug_concept_id,drug_name,approved,immunotherapy,anti_neoplastic,ncbi_gene_id
404,DRD2,hgnc:3023,DRD2,DTC,9/2/20,NaN,0.014977,CLOZAPINE,rxcui:2626,CLOZAPINE,True,False,False,1813
1893,DRD2,hgnc:3023,DRD2,NCI,14-Sep-17,NaN,0.014977,CLOZAPINE,rxcui:2626,CLOZAPINE,True,False,False,1813
1953,MAPK1,hgnc:6871,MAPK1,NCI,14-Sep-17,NaN,0.016669,L-DOPA,rxcui:6375,LEVODOPA,True,False,False,5594
2005,SRC,hgnc:11283,SRC,NCI,14-Sep-17,NaN,0.020049,CLOZAPINE,rxcui:2626,CLOZAPINE,True,False,False,6714
2059,CYP3A43,hgnc:17450,CYP3A43,PharmGKB,4/5/24,NaN,0.077331,clozapine,rxcui:2626,CLOZAPINE,True,False,False,64816
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84434,DBH,hgnc:2689,DBH,PharmGKB,4/5/24,NaN,0.194473,levodopa,rxcui:6375,LEVODOPA,True,False,False,1621
84441,MTHFR,hgnc:7436,MTHFR,PharmGKB,4/5/24,NaN,0.078840,levodopa,rxcui:6375,LEVODOPA,True,False,False,4524
85107,ADORA2A,hgnc:263,ADORA2A,PharmGKB,4/5/24,NaN,0.078840,levodopa,rxcui:6375,LEVODOPA,True,False,False,135
85412,DRD3,hgnc:3024,DRD3,PharmGKB,4/5/24,NaN,0.069455,levodopa,rxcui:6375,LEVODOPA,True,False,False,1814


In [94]:
len(gene_to_index)

124

In [95]:
print(drug_to_index)
print(gene_to_index)
print("Number of relevant drugs: " + str(len(drugs)))
print("Number of relevant genes: " + str(len(genes)))

{'CLOZAPINE': 0, 'LEVODOPA': 1, 'CARBIDOPA ANHYDROUS': 2}
{'1813': 0, '5594': 1, '6714': 2, '64816': 3, '9429': 4, '1543': 5, '12': 6, '3107': 7, '4129': 8, '100': 9, '262': 10, '2952': 11, '3356': 12, '3164': 13, '25970': 14, '718': 15, '7365': 16, '4363': 17, '3127': 18, '2904': 19, '1562': 20, '54658': 21, '1644': 22, '1': 23, '3': 24, '11': 25, '2740': 26, '185': 27, '1742': 28, '2675': 29, '2668': 30, '4524': 31, '1557': 32, '627': 33, '4907': 34, '3350': 35, '4781': 36, '3125': 37, '24': 38, '215': 39, '217': 40, '1812': 41, '1816': 42, '3359': 43, '2944': 44, '1565': 45, '265': 46, '100302144': 47, '2166': 48, '1576': 49, '581': 50, '3363': 51, '100302251': 52, '26': 53, '1128': 54, '28755': 55, '22': 56, '218': 57, '5294': 58, '3358': 59, '11201': 60, '2194': 61, '3558': 62, '1993': 63, '887': 64, '5565': 65, '3115': 66, '4160': 67, '23': 68, '6505': 69, '3952': 70, '3699': 71, '23216': 72, '3600': 73, '5617': 74, '2641': 75, '84062': 76, '3106': 77, '50852': 78, '9135': 79, '7

In [96]:
# Calculate gene degrees in HumanNet
genes_in_humannet = pd.unique(HUMANNET[['Gene1', 'Gene2']].values.ravel())
gene_to_degree = {gene: 0 for gene in genes_in_humannet}

for _, row in HUMANNET.iterrows():
    gene_to_degree[row["Gene1"]] += 1
    gene_to_degree[row["Gene2"]] += 1

gene_to_degree = {str(gene): degree for gene, degree in gene_to_degree.items()}

In [97]:
# Construct gene weight diagonal matrix with 0.01 for genes not in HumanNet
count = 0
gene_weights = np.zeros(len(genes))
index_to_gene = {i: gene for gene, i in gene_to_index.items()}
for index in range(len(genes)):
    gene = index_to_gene[index]
    if gene in gene_to_degree:
        count += 1
        print("FOUND:",gene,gene_to_degree[gene])
        gene_weights[index] = gene_to_degree[gene]
    else:
        print("Not FOUND:",gene)
        gene_weights[index] = 0.01  # Assign a small weight to genes not in HumanNet
diag_gene_weight_matrix = diags(gene_weights,dtype = np.float32)
save_npz(OUTPUT_FOLDER + "diag_gene_weight_matrix.npz", diag_gene_weight_matrix)
print(f"{count} out of {len(genes)} ({count/len(genes)*100:.2f}%) genes are found in HumanNet. {len(genes) - count} out of {len(genes)} ({(len(genes) - count)/len(genes)*100:.2f}%) genes are assigned default weight of 0.01")
print(f"Gene weight diagonal matrix saved as {OUTPUT_FOLDER} + diag_gene_weight_matrix.npz")

FOUND: 1813 47
FOUND: 5594 202
FOUND: 6714 458
Not FOUND: 64816
FOUND: 9429 25
FOUND: 1543 78
Not FOUND: 12
FOUND: 3107 2
FOUND: 4129 22
FOUND: 100 66
FOUND: 262 10
Not FOUND: 2952
FOUND: 3356 88
FOUND: 3164 78
Not FOUND: 25970
FOUND: 718 32
Not FOUND: 7365
FOUND: 4363 14
Not FOUND: 3127
FOUND: 2904 10
FOUND: 1562 10
FOUND: 54658 29
FOUND: 1644 11
Not FOUND: 1
Not FOUND: 3
Not FOUND: 11
FOUND: 2740 82
FOUND: 185 192
FOUND: 1742 51
Not FOUND: 2675
FOUND: 2668 78
FOUND: 4524 22
FOUND: 1557 43
FOUND: 627 11
FOUND: 4907 37
FOUND: 3350 74
FOUND: 4781 41
Not FOUND: 3125
FOUND: 24 7
FOUND: 215 33
FOUND: 217 57
FOUND: 1812 73
FOUND: 1816 60
Not FOUND: 3359
FOUND: 2944 42
FOUND: 1565 52
FOUND: 265 4
Not FOUND: 100302144
FOUND: 2166 2
FOUND: 1576 62
FOUND: 581 357
Not FOUND: 3363
Not FOUND: 100302251
FOUND: 26 14
Not FOUND: 1128
Not FOUND: 28755
Not FOUND: 22
FOUND: 218 52
FOUND: 5294 233
FOUND: 3358 47
Not FOUND: 11201
FOUND: 2194 9
FOUND: 3558 69
Not FOUND: 1993
FOUND: 887 41
Not FOUND: 5565
F

In [114]:
gene_weights

array([1.00e-02, 2.09e+02, 4.70e+01, 1.00e-02, 1.00e+01, 1.00e-02,
       8.80e+01, 5.00e+01, 1.00e-02, 4.90e+01, 4.58e+02, 1.00e-02,
       1.00e-02, 2.50e+01, 4.00e+00, 1.00e-02, 1.20e+01, 1.00e-02,
       2.90e+01, 7.80e+01, 2.70e+01, 1.36e+02, 1.05e+02, 1.00e+02,
       1.00e+00, 1.20e+01, 1.00e-02, 1.66e+02, 1.00e-02, 5.90e+01,
       2.00e+00, 5.20e+01, 1.00e-02, 1.00e-02, 1.00e-02, 1.17e+02,
       3.30e+01, 1.00e-02, 5.00e+00, 1.30e+01, 1.00e-02, 1.00e-02,
       1.00e-02, 5.30e+01, 5.50e+01, 3.70e+01, 3.20e+01, 1.00e-02,
       1.40e+01, 1.00e-02, 1.00e+01, 1.00e+01, 5.50e+01, 7.90e+01,
       7.00e+00, 2.60e+01, 1.00e-02, 1.00e-02, 7.40e+01, 1.80e+01,
       2.30e+01, 2.54e+02, 1.06e+02, 5.05e+02, 1.00e-02, 8.20e+01,
       1.52e+02, 1.00e+00, 1.00e-02, 7.80e+01, 2.20e+01, 4.30e+01,
       4.00e+01, 1.87e+02, 5.90e+01, 3.70e+01, 1.00e-02, 6.80e+01,
       4.10e+01, 1.00e-02, 2.90e+01, 6.80e+01, 1.00e-02, 7.30e+01,
       7.00e+01, 3.40e+01, 7.00e+00, 5.00e+01, 4.70e+01, 7.00e

In [115]:
# Add degrees to DGIDB with fallback to 0.01 for missing genes
relevant_rows['degree'] = relevant_rows['ncbi_gene_id'].map(gene_to_degree).fillna(0.01)

In [116]:
relevant_rows

,gene_claim_name,gene_concept_id,gene_name,interaction_source_db_name,interaction_source_db_version,interaction_type,interaction_score,drug_claim_name,drug_concept_id,drug_name,approved,immunotherapy,anti_neoplastic,ncbi_gene_id,degree
74,NCBIGENE:926,hgnc:1707,CD8B,GuideToPharmacology,2024.1,inhibitor,0.083478,IUPHAR.LIGAND:7135,rxcui:203204,BUPROPION HYDROCHLORIDE,True,False,False,926,0.01
207,TNIK,hgnc:30765,TNIK,PharmGKB,4/5/24,NaN,0.391848,risperidone,rxcui:35636,RISPERIDONE,True,False,False,23043,209.00
404,DRD2,hgnc:3023,DRD2,DTC,9/2/20,NaN,0.014977,CLOZAPINE,rxcui:2626,CLOZAPINE,True,False,False,1813,47.00
430,FAM178B,hgnc:28036,FAM178B,PharmGKB,4/5/24,NaN,2.500365,lithium,rxcui:6448,LITHIUM,True,False,False,51252,0.01
497,NCBIGENE:262,hgnc:457,AMD1,GuideToPharmacology,2024.1,inhibitor,0.016765,IUPHAR.LIGAND:50,rxcui:221153,QUETIAPINE FUMARATE,True,False,False,262,10.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88064,HTT,hgnc:4851,HTT,PharmGKB,4/5/24,NaN,0.010380,risperidone,rxcui:35636,RISPERIDONE,True,False,False,3064,155.00
88065,TJP1,hgnc:11827,TJP1,PharmGKB,4/5/24,NaN,0.391848,risperidone,rxcui:35636,RISPERIDONE,True,False,False,7082,61.00
88066,PPA2,hgnc:28883,PPA2,PharmGKB,4/5/24,NaN,1.567393,risperidone,rxcui:35636,RISPERIDONE,True,False,False,27068,18.00
88263,NCBIGENE:3,hgnc:8,A2MP1,GuideToPharmacology,2024.1,agonist,0.017364,IUPHAR.LIGAND:50,rxcui:221153,QUETIAPINE FUMARATE,True,False,False,3,0.01


In [117]:
num_of_irre_degree = len(relevant_rows[relevant_rows['degree'] == 0.01])
num_relevant_rows_entries = len(relevant_rows)
print(f"Number of terms with filled degree (0.01): {num_of_irre_degree}")
print(f"Percentage of terms with filled degree (0.01): {num_of_irre_degree / num_relevant_rows_entries}")

Number of terms with filled degree (0.01): 213
Percentage of terms with filled degree (0.01): 0.2642679900744417


In [118]:
print(len(genes), len(drugs))

359 16


In [119]:
# Initialize a sparse incidence matrix
incidence_matrix = dok_matrix((len(genes), len(drugs)), dtype=np.float32)
binary_incidence_matrix = dok_matrix((len(genes), len(drugs)), dtype=int)

# Initialize degree diagonal matrix
hypernode_degree = np.zeros(len(genes))
hyperedge_degree = np.zeros(len(drugs))
hyperedge_degree_weightless = np.zeros(len(drugs))

In [120]:
relevant_rows.columns

Index(['gene_claim_name', 'gene_concept_id', 'gene_name',
       'interaction_source_db_name', 'interaction_source_db_version',
       'interaction_type', 'interaction_score', 'drug_claim_name',
       'drug_concept_id', 'drug_name', 'approved', 'immunotherapy',
       'anti_neoplastic', 'ncbi_gene_id', 'degree'],
      dtype='object')

In [121]:
# Populate the matrices by processing the relevant rows in DGIDB
i = 0
repeated_rows = []
for _, row in relevant_rows.iterrows():
    gene_idx = gene_to_index[row['ncbi_gene_id']]
    drug_idx = drug_to_index[row['drug_name']]
    
    if (incidence_matrix[gene_idx, drug_idx] != 0):
        repeated_rows.append((row['ncbi_gene_id'], row['drug_name'],i))
    else:
        hypernode_degree[gene_idx] += 1
        hyperedge_degree[drug_idx] += row['degree']
        hyperedge_degree_weightless[drug_idx] += 1
        incidence_matrix[gene_idx, drug_idx] = row['degree']
        binary_incidence_matrix[gene_idx, drug_idx] = 1
    i += 1

In [122]:
# Sanity Cheeeeeeeeeeeeeeeeck
print(binary_incidence_matrix.shape)
print(binary_incidence_matrix.nnz)
print(len(repeated_rows))
print(len(relevant_rows), "(Should be the sum of the two numbers above)")

(359, 16)
632
174
806 (Should be the sum of the two numbers above)


In [123]:
len(hypernode_degree)

359

In [124]:
# # Show nonzero rows sum
# row_sums = np.sum(incidence_matrix.T, axis=1)
# nonzero_row_sums = row_sums[row_sums != 0]
# print(nonzero_row_sums)
# print(hyperedge_degree[hyperedge_degree.nonzero()])

# Build inverse diagonal degree matrix
diag_node_degree_matrix = diags(hypernode_degree,dtype = np.float32)
inverse_hypernode_degrees = np.reciprocal(hypernode_degree, where=hypernode_degree!=0,dtype = np.float32)
inverse_diag_node_degree_matrix = diags(inverse_hypernode_degrees,dtype = np.float32)

inverse_hyperedge_degrees = np.reciprocal(hyperedge_degree, where=hyperedge_degree!=0,dtype = np.float32)
inverse_diag_edge_degree_matrix = diags(inverse_hyperedge_degrees,dtype = np.float32)

inverse_hyperedge_degrees_weightless = np.reciprocal(hyperedge_degree_weightless, where=hyperedge_degree_weightless!=0,dtype = np.float32)
inverse_diag_edge_degree_weightless_matrix = diags(inverse_hyperedge_degrees_weightless,dtype = np.float32)

# Convert the DOK matrix to CSR format
incidence_matrix = incidence_matrix.tocsr()
binary_incidence_matrix = binary_incidence_matrix.tocsr()


# Save the matrix as .npz file
save_npz(OUTPUT_FOLDER + "hypergraph_incidence_matrix_weighted.npz", incidence_matrix)
save_npz(OUTPUT_FOLDER + "hypergraph_incidence_matrix_binary.npz", binary_incidence_matrix)
save_npz(OUTPUT_FOLDER + "diag_node_degree_matrix.npz", diag_node_degree_matrix)
save_npz(OUTPUT_FOLDER + "inverse_diag_node_degree_matrix.npz", inverse_diag_node_degree_matrix)
save_npz(OUTPUT_FOLDER + "inverse_diag_edge_degree_matrix.npz", inverse_diag_edge_degree_matrix)
save_npz(OUTPUT_FOLDER + "inverse_diag_edge_degree_weightless_matrix.npz", inverse_diag_edge_degree_weightless_matrix)
# Print confirmation
print(f"Weighted incidence matrix saved as {OUTPUT_FOLDER}hypergraph_incidence_matrix_weighted.npz'.")
print(f"Binary incidence matrix saved as {OUTPUT_FOLDER}hypergraph_incidence_matrix_binary.npz'.")
print(f"Diagonal node degree matrix saved as {OUTPUT_FOLDER}inverse_diag_node_degree_matrix.npz'.")
print(f"Inverse diagonal node degree matrix saved as {OUTPUT_FOLDER}inverse_diag_node_degree_matrix.npz'.")
print(f"Inverse diagonal edge degree matrix saved as {OUTPUT_FOLDER}inverse_diag_edge_degree_matrix.npz'.")
print(f"Inverse diagonal edge degree weightless matrix saved as {OUTPUT_FOLDER}inverse_diag_edge_degree_weightless_matrix.npz'.")


Weighted incidence matrix saved as ./output/DGIDB_BIPOLAR/hypergraph_incidence_matrix_weighted.npz'.
Binary incidence matrix saved as ./output/DGIDB_BIPOLAR/hypergraph_incidence_matrix_binary.npz'.
Diagonal node degree matrix saved as ./output/DGIDB_BIPOLAR/inverse_diag_node_degree_matrix.npz'.
Inverse diagonal node degree matrix saved as ./output/DGIDB_BIPOLAR/inverse_diag_node_degree_matrix.npz'.
Inverse diagonal edge degree matrix saved as ./output/DGIDB_BIPOLAR/inverse_diag_edge_degree_matrix.npz'.
Inverse diagonal edge degree weightless matrix saved as ./output/DGIDB_BIPOLAR/inverse_diag_edge_degree_weightless_matrix.npz'.


In [125]:
print(incidence_matrix)
print(binary_incidence_matrix)
print(diag_node_degree_matrix)
print(inverse_diag_node_degree_matrix)
print(inverse_diag_edge_degree_matrix)
print(inverse_diag_edge_degree_weightless_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 632 stored elements and shape (359, 16)>
  Coords	Values
  (0, 0)	0.009999999776482582
  (0, 4)	0.009999999776482582
  (1, 1)	209.0
  (2, 2)	47.0
  (2, 7)	47.0
  (2, 4)	47.0
  (2, 1)	47.0
  (2, 6)	47.0
  (2, 8)	47.0
  (2, 12)	47.0
  (2, 14)	47.0
  (2, 3)	47.0
  (3, 3)	0.009999999776482582
  (4, 4)	10.0
  (4, 2)	10.0
  (4, 12)	10.0
  (4, 14)	10.0
  (4, 7)	10.0
  (4, 1)	10.0
  (4, 8)	10.0
  (4, 6)	10.0
  (5, 5)	0.009999999776482582
  (5, 13)	0.009999999776482582
  (6, 6)	88.0
  (6, 2)	88.0
  :	:
  (335, 1)	0.009999999776482582
  (336, 13)	0.009999999776482582
  (337, 11)	32.0
  (338, 4)	0.009999999776482582
  (339, 7)	54.0
  (340, 13)	72.0
  (340, 11)	72.0
  (341, 1)	0.009999999776482582
  (342, 1)	0.009999999776482582
  (343, 1)	69.0
  (344, 1)	113.0
  (345, 11)	42.0
  (346, 11)	413.0
  (347, 13)	45.0
  (348, 4)	115.0
  (349, 4)	80.0
  (350, 1)	244.0
  (351, 0)	0.009999999776482582
  (352, 11)	138.0
  (353, 7)	249.0
  (354, 1

In [126]:
print(incidence_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 632 stored elements and shape (359, 16)>
  Coords	Values
  (0, 0)	0.009999999776482582
  (0, 4)	0.009999999776482582
  (1, 1)	209.0
  (2, 2)	47.0
  (2, 7)	47.0
  (2, 4)	47.0
  (2, 1)	47.0
  (2, 6)	47.0
  (2, 8)	47.0
  (2, 12)	47.0
  (2, 14)	47.0
  (2, 3)	47.0
  (3, 3)	0.009999999776482582
  (4, 4)	10.0
  (4, 2)	10.0
  (4, 12)	10.0
  (4, 14)	10.0
  (4, 7)	10.0
  (4, 1)	10.0
  (4, 8)	10.0
  (4, 6)	10.0
  (5, 5)	0.009999999776482582
  (5, 13)	0.009999999776482582
  (6, 6)	88.0
  (6, 2)	88.0
  :	:
  (335, 1)	0.009999999776482582
  (336, 13)	0.009999999776482582
  (337, 11)	32.0
  (338, 4)	0.009999999776482582
  (339, 7)	54.0
  (340, 13)	72.0
  (340, 11)	72.0
  (341, 1)	0.009999999776482582
  (342, 1)	0.009999999776482582
  (343, 1)	69.0
  (344, 1)	113.0
  (345, 11)	42.0
  (346, 11)	413.0
  (347, 13)	45.0
  (348, 4)	115.0
  (349, 4)	80.0
  (350, 1)	244.0
  (351, 0)	0.009999999776482582
  (352, 11)	138.0
  (353, 7)	249.0
  (354, 1

In [127]:
print(np.sum(gene_weights != 0.01), "out of", len(gene_weights), "genes have non-default weights (aka are found in HumanNet).")

246 out of 359 genes have non-default weights (aka are found in HumanNet).


In [128]:
row_sums = np.sum(np.abs(binary_incidence_matrix), axis=1)

# Indices of zero rows
zero_row_indices = np.where(row_sums == 0)[0]

# Count
num_zero_rows = len(zero_row_indices)

print("Zero row indices:", zero_row_indices)
print("Number of zero rows:", num_zero_rows)

Zero row indices: []
Number of zero rows: 0


In [129]:
for drug in specific_disease_drugs:
    if drug in drug_to_index:
        idx = drug_to_index[drug]
        print(f"Drug: {drug}, Index: {idx}")
    else:
        print(f"Drug: {drug} not found in drug_to_index.")


Drug: CLONAZEPAM, Index: 15
Drug: BUPROPION HYDROCHLORIDE, Index: 0
Drug: SERTRALINE HYDROCHLORIDE, Index: 9
Drug: OLANZAPINE, Index: 6
Drug: ZIPRASIDONE, Index: 12
Drug: QUETIAPINE FUMARATE, Index: 4
Drug: CHLORPROMAZINE, Index: 8
Drug: CARBAMAZEPINE, Index: 13
Drug: ALLOPURINOL, Index: 5
Drug: PERPHENAZINE, Index: 7
Drug: VALPROIC ACID, Index: 11
Drug: CLOZAPINE, Index: 2
Drug: LITHIUM, Index: 3
Drug: RISPERIDONE, Index: 1
Drug: LAMOTRIGINE, Index: 10
Drug: ARIPIPRAZOLE LAUROXIL, Index: 14


In [130]:
# # Compute gene-gene adjacency matrix by projecting via shared drugs
# adj_matrix = adj_matrix = binary_csr_matrix @ binary_csr_matrix.T  # Matrix multiplication: shared drugs
# adj_matrix.setdiag(0)
# adj_matrix.eliminate_zeros()

# # --- Step 2: Extract Edgelist from Upper Triangle Only ---
# # Use sparse coo_matrix to iterate efficiently
# from scipy.sparse import triu

# adj_matrix_upper = triu(adj_matrix, k=1)  # upper triangle, no diag
# adj_coo = adj_matrix_upper.tocoo()

# # Optional: if you have gene names
# # gene_names = ['TP53', 'EGFR', 'BRCA1', ...]
# # Otherwise use indices as names

# edges = []
# for i, j, v in zip(adj_coo.row, adj_coo.col, adj_coo.data):
#     edges.append((i, j, v))  # replace i/j with gene_names[i] if available

# # Convert to DataFrame and save
# edge_df = pd.DataFrame(edges, columns=["Gene1", "Gene2", "Weight"])

# # If you have gene names, map them:
# # edge_df["Gene1"] = edge_df["Gene1"].map(lambda i: gene_names[i])
# # edge_df["Gene2"] = edge_df["Gene2"].map(lambda i: gene_names[i])

# edge_df.to_csv("gene_gene_edgelist.csv", index=False)

In [131]:
# import pandas as pd

# if 'NCBI_INFO' not in globals():
#     print("Reading gene2refseq.gz...")
#     NCBI_INFO = pd.read_csv("../Data/ncbi/gene2refseq.gz", sep='\t', compression='gzip')
# else:
#     print("NCBI_INFO already loaded.")

In [132]:
# index_to_ncbi = {idx: gene for gene, idx in gene_to_index.items()}
# human_gene2refseq = NCBI_INFO[NCBI_INFO['#tax_id'] == 9606]
# id_to_gene_claim = pd.Series(human_gene2refseq.Symbol.values, index=human_gene2refseq.GeneID).to_dict()

# # Your existing function to get common gene name from ncbi gene id
# def get_gene_claim_name(ncbi_gene_id):
#     try:
#         ncbi_gene_id = int(ncbi_gene_id)
#         result = id_to_gene_claim.get(ncbi_gene_id, None)
#         return result if result else "Gene name not found"
#     except:
#         return "Gene name not found"

In [133]:
# # Step 1: Map index → NCBI gene ID
# edge_df['Gene1_ncbi'] = edge_df['Gene1'].map(index_to_ncbi)
# edge_df['Gene2_ncbi'] = edge_df['Gene2'].map(index_to_ncbi)

# # Step 2: Map NCBI gene ID → gene symbol
# edge_df['Gene1'] = edge_df['Gene1_ncbi'].apply(get_gene_claim_name)
# edge_df['Gene2'] = edge_df['Gene2_ncbi'].apply(get_gene_claim_name)

# # Step 3: Drop temp NCBI ID columns
# edge_df = edge_df.drop(columns=['Gene1_ncbi', 'Gene2_ncbi'])

# # Optional: Save to CSV
# edge_df.to_csv('gene_gene_edgelist_named.csv', index=False)
